# Entrenamiento de Modelos Predictivos

En este notebook se inicia la etapa de modelado del proyecto de predicción de cancelación de clientes en TelecomX.

A partir del dataset previamente procesado, se realizará la separación entre variables predictoras y variable objetivo, así como la división de los datos en conjuntos de entrenamiento y prueba.

Posteriormente, se entrenarán modelos de clasificación con el propósito de predecir la cancelación de clientes (*churn*) y comparar su desempeño en etapas posteriores del proyecto.

## Objetivos

- Cargar el dataset preprocesado
- Separar las variables predictoras y la variable objetivo
- Dividir los datos en entrenamiento y prueba
- Entrenar modelos de clasificación
- Dejar preparados los modelos para su evaluación

##Montamos el Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Importamos las librerías necesarias

In [2]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

##Definimos las rutas del proyecto

In [3]:
PROJECT_PATH = "/content/drive/MyDrive/TelecomX_Churn_ML"
DATA_PATH = os.path.join(PROJECT_PATH, "data", "processed")

dataset_path = os.path.join(DATA_PATH, "TelecomX_model_ready.csv")

print("Ruta del dataset:", dataset_path)

Ruta del dataset: /content/drive/MyDrive/TelecomX_Churn_ML/data/processed/TelecomX_model_ready.csv


##Cargamos el dataset

In [4]:
df = pd.read_csv(dataset_path)

df.head()

,abandono,adulto_mayor,tiene_pareja,tiene_dependientes,antiguedad_meses,servicio_telefono,lineas_multiples,proteccion_dispositivo,soporte_tecnico,streaming_tv,...,servicio_internet_No,seguridad_online_No internet service,seguridad_online_Yes,respaldo_online_No internet service,respaldo_online_Yes,tipo_contrato_One year,tipo_contrato_Two year,metodo_pago_Credit card (automatic),metodo_pago_Electronic check,metodo_pago_Mailed check
0,0,0,1,1,9,1,0,0,1,1,...,False,False,False,False,True,True,False,False,False,True
1,0,0,0,0,9,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,True
2,1,0,0,0,4,1,0,1,0,0,...,False,False,False,False,False,False,False,False,True,False
3,1,1,1,0,13,1,0,1,0,1,...,False,False,False,False,True,False,False,False,True,False
4,1,1,1,0,3,1,0,0,1,1,...,False,False,False,False,False,False,False,False,False,True


##Convertimos las columnas que tienen valores de True/False en 0 y 1


In [5]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

In [6]:
df.head()

,abandono,adulto_mayor,tiene_pareja,tiene_dependientes,antiguedad_meses,servicio_telefono,lineas_multiples,proteccion_dispositivo,soporte_tecnico,streaming_tv,...,servicio_internet_No,seguridad_online_No internet service,seguridad_online_Yes,respaldo_online_No internet service,respaldo_online_Yes,tipo_contrato_One year,tipo_contrato_Two year,metodo_pago_Credit card (automatic),metodo_pago_Electronic check,metodo_pago_Mailed check
0,0,0,1,1,9,1,0,0,1,1,...,0,0,0,0,1,1,0,0,0,1
1,0,0,0,0,9,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,1,0,0,0,4,1,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,1,1,0,13,1,0,1,0,1,...,0,0,0,0,1,0,0,0,1,0
4,1,1,1,0,3,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,1


## Separación de Datos

En esta etapa se divide el dataset en dos conjuntos:

- **Conjunto de entrenamiento (70%)**: utilizado para entrenar los modelos de Machine Learning.
- **Conjunto de prueba (30%)**: utilizado para evaluar el desempeño del modelo en datos no vistos.

Se utiliza una división estratificada para mantener la misma proporción de cancelación de clientes en ambos conjuntos.

In [7]:
X = df.drop("abandono", axis=1)
y = df["abandono"]

##Dividimos el dataset

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [9]:
print("Dimensiones de entrenamiento:")
print(X_train.shape)

print("\nDimensiones de prueba:")
print(X_test.shape)

Dimensiones de entrenamiento:
(4922, 26)

Dimensiones de prueba:
(2110, 26)


In [10]:
print("Proporción de abandono en entrenamiento:")
print(y_train.value_counts(normalize=True))

print("\nProporción de abandono en prueba:")
print(y_test.value_counts(normalize=True))

Proporción de abandono en entrenamiento:
abandono
0    0.734254
1    0.265746
Name: proportion, dtype: float64

Proporción de abandono en prueba:
abandono
0    0.734123
1    0.265877
Name: proportion, dtype: float64


## Conclusión

El dataset fue dividido en conjuntos de entrenamiento y prueba utilizando una proporción 70/30.  
Se aplicó estratificación para mantener la distribución original de la variable objetivo `abandono` en ambos conjuntos.

Esta división permitirá evaluar el desempeño de los modelos de Machine Learning utilizando datos que no fueron utilizados durante el entrenamiento.

## Creación de modelos

En esta etapa se entrenarán dos modelos de clasificación para predecir la cancelación de clientes.

Se seleccionaron dos enfoques diferentes:

- **Regresión Logística**, como modelo lineal e interpretable, sensible a la escala de los datos.
- **Random Forest**, como modelo basado en árboles, capaz de capturar relaciones no lineales y no sensible a la escala de las variables.

Para el caso de la **Regresión Logística**, se aplicará estandarización a las variables numéricas con el fin de evitar que diferencias de magnitud entre variables influyan de manera desproporcionada en el entrenamiento del modelo.

En cambio, **Random Forest** será entrenado sin normalización, ya que este tipo de algoritmo no depende de la escala de los datos.

##Importamos la librerías necesarias

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

##Definición de las columnas continuas

In [12]:
columnas_continuas = ["antiguedad_meses", "cargo_mensual", "cargo_total", "cargo_diario"]

##Creamos copias para entrenamiento y prueba para regresión logística

In [13]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

##Estandarizamos las columnas continuas

In [14]:
scaler = StandardScaler()

X_train_scaled[columnas_continuas] = scaler.fit_transform(X_train[columnas_continuas])
X_test_scaled[columnas_continuas] = scaler.transform(X_test[columnas_continuas])

##Entrenamos un modelo de regresión logística

In [16]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)

print("Modelo de Regresión Logística entrenado correctamente.")

Modelo de Regresión Logística entrenado correctamente.


##Entrenamos modelo Random Forest

In [17]:
random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train, y_train)

print("Modelo Random Forest entrenado correctamente.")

Modelo Random Forest entrenado correctamente.


##Realizamos predicciones de los modelos

In [20]:
y_pred_logistic = logistic_model.predict(X_test_scaled)
y_pred_rf = random_forest_model.predict(X_test)


In [21]:
predicciones = pd.DataFrame({
    "Real": y_test,
    "LogisticRegression": y_pred_logistic,
    "RandomForest": y_pred_rf
})

predicciones.head(10)

,Real,LogisticRegression,RandomForest
4281,0,0,0
1832,0,0,1
2402,0,0,0
5506,0,0,0
1791,0,0,0
2711,0,0,0
5662,0,0,0
4857,0,0,0
5148,0,0,0
4915,1,0,0


In [22]:
pd.Series(y_pred_logistic).value_counts()

,count
0,1650
1,460


In [23]:
pd.Series(y_pred_rf).value_counts()

,count
0,1676
1,434


Las predicciones generadas por ambos modelos muestran que los dos algoritmos identifican tanto clientes que permanecen como clientes que cancelan el servicio.

La Regresión Logística predice ligeramente más casos de cancelación que el modelo Random Forest, lo cual sugiere que podría tener una mayor sensibilidad para detectar clientes con riesgo de churn.

Sin embargo, para determinar cuál modelo tiene mejor desempeño será necesario evaluar métricas de clasificación como accuracy, precision, recall y F1-score en la siguiente etapa del proyecto.